### Criação do schema Silver

Cria o banco de dados silver. As tabelas Bronze não são alteradas — cada tabela Silver é reconstruída a partir da Bronze aplicando renomeação de colunas para português, tipagem correta e as regras de qualidade de dados específicas de cada tabela.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

### silver.tb_info_filmes

Origem: bronze.tb_movies_info. Renomeia as colunas para português. Normaliza a coluna de status antes de traduzir os valores para português. Deriva ano_lancamento a partir da data. Por fim, deduplica por id_filme usando uma janela ordenada por ingestion_datetime decrescente, mantendo sempre o registro mais recente isso garante unicidade por filme mesmo que a Bronze tenha registros repetidos.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.tb_movies_info")

df = (
    df.withColumnRenamed("id", "id_filme")
      .withColumnRenamed("title", "titulo")
      .withColumnRenamed("original_title", "titulo_original")
      .withColumnRenamed("release_date", "data_lancamento_raw")
      .withColumnRenamed("runtime", "duracao_minutos")
      .withColumnRenamed("original_language", "idioma_original")
      .withColumnRenamed("status", "status_raw")
      .withColumnRenamed("overview", "sinopse")
      .withColumnRenamed("tagline", "frase_divulgacao")
)

status_normalizado = F.upper(F.trim(F.regexp_replace(F.col("status_raw"), r"[-_]+", " ")))
status_normalizado = F.trim(F.regexp_replace(status_normalizado, r"\s+", " "))
df = df.withColumn("status_normalizado", status_normalizado)

df = df.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "RELEASED", "Lançado")
     .when(F.col("status_normalizado") == "POST PRODUCTION", "Pós-Produção")
     .when(F.col("status_normalizado") == "IN PRODUCTION", "Em Produção")
     .when(F.col("status_normalizado") == "PLANNED", "Planejado")
     .when(F.col("status_normalizado") == "RUMORED", "Rumores")
     .when(F.col("status_normalizado").isin("CANCELED", "CANCELLED"), "Cancelado")
     .otherwise("Não Informado")
)

df = df.withColumn(
    "data_lancamento",
    F.coalesce(
        F.try_to_date("data_lancamento_raw", F.lit("yyyy-MM-dd")),
        F.try_to_date("data_lancamento_raw", F.lit("dd/MM/yyyy")),
        F.try_to_date("data_lancamento_raw", F.lit("MM/dd/yyyy")),
        F.try_to_date("data_lancamento_raw", F.lit("dd-MM-yyyy")),
        F.try_to_date("data_lancamento_raw", F.lit("MM-dd-yyyy")),
    )
)

df = df.withColumn("ano_lancamento", F.year("data_lancamento"))

janela = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(janela)).filter("rn = 1").drop("rn")

df_silver_info = df.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"
)

df_silver_info.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_info_filmes")

display(df_silver_info)

### silver.tb_cotacao_dolar

A API do Banco Central não retorna cotação em finais de semana/feriados, então a série de datas tem buracos. Aqui, construo um calendário contínuo e faço um left join com as cotações reais. Os dias sem cotação ficam com valor NULL após o join; aplico então a
técnica de Forward Fill usando uma window function ordenada por data, que propaga o último valor válido conhecido para os
dias seguintes sem cotação.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cambio = spark.table("workspace.bronze.tb_cotacao_dolar")

df_cambio = (
    df_cambio.withColumn("data_cotacao", F.to_date("dataHoraCotacao"))
             .withColumnRenamed("cotacaoCompra", "valor_dolar")
             .select("data_cotacao", "valor_dolar")
             .dropDuplicates(["data_cotacao"])
)

data_min, data_max = df_cambio.selectExpr("min(data_cotacao)", "max(data_cotacao)").first()

df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) AS data_cotacao
""")

df_completo = df_calendario.join(df_cambio, "data_cotacao", "left")

janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
df_silver_cambio = df_completo.withColumn(
    "valor_dolar",
    F.last("valor_dolar", ignorenulls=True).over(janela_ffill)
)

df_silver_cambio.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_cotacao_dolar")

display(df_silver_cambio)

### silver.tb_financeiro_filmes

Antes de tudo, deduplica por id_filme. A função limpar_e_converter() centraliza toda a regra de limpeza de orçamento e receita: trata textos de ausência de dado; detecta abreviações de escala (K = mil, M = milhão, B = bilhão) ANTES de remover as letras, aplicando o
multiplicador correto; remove símbolos de moeda e pontuação de milhar; e trata valores zerados ou negativos como ausentes. Os valores em
Reais são calculados aplicando a cotação mais recente disponível na tb_cotacao_dolar.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.tb_movies_financials")

df = df.withColumnRenamed("id", "id_filme")

window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

taxa_dolar = (
    spark.table("workspace.silver.tb_cotacao_dolar")
    .orderBy(F.col("data_cotacao").desc())
    .select("valor_dolar")
    .first()["valor_dolar"]
)
print(f"Taxa de conversão USD->BRL utilizada: {taxa_dolar}")

def limpar_e_converter(col_raw):
    tratado = F.when(
        F.trim(F.upper(col_raw)).isin("UNKNOWN", "NÃO INFORMADO", "N/A", "NA", "NULL", ""),
        None
    ).otherwise(col_raw)

    tratado_upper = F.trim(F.upper(tratado))

    multiplicador = (
        F.when(tratado_upper.rlike("[0-9](K)$"), F.lit(1000))
         .when(tratado_upper.rlike("[0-9](M)$"), F.lit(1000000))
         .when(tratado_upper.rlike("[0-9](B)$"), F.lit(1000000000))
         .otherwise(F.lit(1))
    )

    limpo = F.regexp_replace(tratado, r"[^0-9.\-]", "")
    limpo = F.when(limpo == "", None).otherwise(limpo)

    valor_base = limpo.try_cast("decimal(18,4)")
    valor = F.round(valor_base * multiplicador, 2).try_cast("decimal(18,2)")

    return F.when(valor <= 0, None).otherwise(valor)

df = (
    df.withColumn("orcamento_usd", limpar_e_converter(F.col("budget")))
      .withColumn("receita_usd", limpar_e_converter(F.col("revenue")))
)

df = (
    df.withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(taxa_dolar), 2))
      .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(taxa_dolar), 2))
)

df = (
    df.withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
      .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
      .withColumn(
          "margem_lucro_percentual",
          F.round(F.try_divide(F.col("lucro_usd"), F.col("receita_usd")) * 100, 2)
      )
)

df_silver_financeiro = df.select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual"
)

df_silver_financeiro.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_financeiro_filmes")

display(df_silver_financeiro)

### silver.tb_metricas_engajamento

Deduplica por id_filme da mesma forma que a tabela financeira. A coluna de popularidade passou por 3 camadas de limpeza:
(1) se o valor bruto contém qualquer letra, é texto de sinopse/tag vazado por Column Shift, não um número, vira NULL; (2) um valor numérico inteiro "redondo" dentro da faixa de anos de lançamento plausíveis (1870–2030) tem mais cara de ano vazado do que de popularidade real também vira NULL; (3) por fim, aplico o teto de sanidade de 0 a 10.000 como última barreira de defesa.Por fim, aplico os limites de
negócio: notas fora de 0–10 e contagens/popularidade negativas viram NULL.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.tb_movies_metrics")

df = df.withColumnRenamed("id", "id_filme")

window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

# Camada 1:
popularidade_bruta = F.col("popularity")
eh_texto = popularidade_bruta.rlike("(?i)[a-zà-ÿ]")

popularidade_limpa = F.regexp_replace(popularidade_bruta, ",", ".")
popularidade_limpa = F.regexp_replace(popularidade_limpa, r"[^0-9.\-]", "")
df = df.withColumn(
    "popularidade",
    F.when(eh_texto, None).otherwise(popularidade_limpa.try_cast("double"))
)

# Camada 2:
eh_ano_vazado = (F.col("popularidade") == F.floor(F.col("popularidade"))) & (F.col("popularidade").between(1870, 2030))
df = df.withColumn("popularidade", F.when(eh_ano_vazado, None).otherwise(F.col("popularidade")))

df = (
    df.withColumn("nota_media_tmdb", F.col("vote_average").try_cast("double"))
      .withColumn("qtd_votos_tmdb", F.col("vote_count").try_cast("int"))
      .withColumn("nota_media_imdb", F.col("averageRating").try_cast("double"))
      .withColumn("qtd_votos_imdb", F.col("numVotes").try_cast("int"))
)

# Camada 3:
df = (
    df.withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
      .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
      .withColumn("popularidade", F.when(F.col("popularidade").between(0, 10000), F.col("popularidade")))
      .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
      .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
)

df_silver_metricas = df.select(
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
)

df_silver_metricas.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_metricas_engajamento")

display(df_silver_metricas)

### silver.tb_avaliacoes_usuarios

Renomeia as colunas e remove registros integralmente duplicados com dropDuplicates. Valida que a nota do usuário respeita a escala 0–10, descartando (NULL) qualquer valor fora dela. Usa F.coalesce(comentario, F.lit("")) antes de checar se o texto é vazio/só espaços — isso garante que tanto um comentário NULL quanto um comentário em branco sejam padronizados com o texto "Sem comentário".

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.tb_movies_reviews")

df = (
    df.withColumnRenamed("id", "id_filme")
      .withColumnRenamed("nome", "nome_usuario")
      .withColumnRenamed("nota", "nota_usuario_raw")
      .withColumnRenamed("comentario", "comentario_usuario_raw")
)

df = df.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario_raw", "comentario_usuario_raw"])

df = df.withColumn("nota_usuario", F.col("nota_usuario_raw").try_cast("double"))
df = df.withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))

df = df.withColumn(
    "comentario_usuario",
    F.when(
        F.trim(F.coalesce(F.col("comentario_usuario_raw"), F.lit(""))) == "",
        F.lit("Sem comentário")
    ).otherwise(F.col("comentario_usuario_raw"))
)

df_silver_avaliacoes = df.select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")

df_silver_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_avaliacoes_usuarios")

display(df_silver_avaliacoes)

### silver.tb_generos

Investigando os dados brutos, descobri que o delimitador real usado na coluna não é vírgula ou ponto e vírgula como o enunciado sugeria, e sim (pipe), por isso, primeiro eu normalizo qualquer vírgula/ponto e vírgula remanescente para, e depois faço o explode(). Depois do explode, em vez de tentar filtrar lixo com heurísticas de tamanho/regex comparo cada valor contra uma whitelist fechada com os 19 gêneros padrão do TMDB.

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.tb_credits_and_tags")
df = df.withColumnRenamed("id", "id_filme")

generos_normalizado = F.regexp_replace(F.col("genres"), "[,;]", "|")

df_generos = df.withColumn("genero_raw", F.explode(F.split(generos_normalizado, r"\|")))

df_generos = df_generos.withColumn(
    "nome_genero",
    F.trim(F.regexp_replace(F.col("genero_raw"), '"', ''))
)

generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

df_generos = df_generos.filter(F.col("nome_genero").isin(generos_validos))

df_silver_generos = df_generos.select("id_filme", "nome_genero").dropDuplicates()

df_silver_generos.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_generos")

display(df_silver_generos.select("nome_genero").distinct().orderBy("nome_genero"))

### silver.tb_pessoas_empresas

Como as 4 colunas passam pela mesma lógica de limpeza, criei uma função reutilizável extrair_entidades() para evitar repetir o código 4 vezes. Ela normaliza os delimitadores, faz o explode, padroniza a capitalização com initcap(), e filtra resíduos, e caminhos de imagem que vazaram da base. A função é chamada uma vez para cada uma das 4 colunas de origem, mapeando para o tipo_entidade correspondente. Os 4 resultados são unidos com unionByName e deduplicados antes da gravação final.

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.tb_credits_and_tags")
df = df.withColumnRenamed("id", "id_filme")

def extrair_entidades(df_origem, coluna_origem, tipo_entidade):
    normalizado = F.regexp_replace(F.col(coluna_origem), "[,;]", "|")

    exploded = df_origem.select(
        "id_filme",
        F.explode(F.split(normalizado, r"\|")).alias("nome_raw")
    )

    exploded = exploded.withColumn(
        "nome_pessoa_empresa",
        F.initcap(F.trim(F.regexp_replace(F.col("nome_raw"), '"', '')))
    )

    exploded = exploded.filter(
        (F.col("nome_pessoa_empresa") != "") &
        (F.length(F.col("nome_pessoa_empresa")) <= 50) &
        (~F.col("nome_pessoa_empresa").rlike(r"^[0-9.\-]+$")) &
        (~F.col("nome_raw").contains("/")) &
        (~F.lower(F.col("nome_raw")).rlike(r"\.(jpg|jpeg|png|gif)$"))
    )

    return exploded.withColumn("tipo_entidade", F.lit(tipo_entidade))

df_atores = extrair_entidades(df, "cast", "Ator")
df_diretores = extrair_entidades(df, "directors", "Diretor")
df_roteiristas = extrair_entidades(df, "writers", "Roteirista")
df_produtoras = extrair_entidades(df, "production_companies", "Produtora")

df_pessoas_empresas = df_atores.unionByName(df_diretores).unionByName(df_roteiristas).unionByName(df_produtoras)

df_silver_pessoas_empresas = (
    df_pessoas_empresas
    .select("id_filme", "nome_pessoa_empresa", "tipo_entidade")
    .dropDuplicates(["id_filme", "nome_pessoa_empresa", "tipo_entidade"])
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_pessoas_empresas")

display(df_silver_pessoas_empresas)